<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 5 (1): Tools & Agents — Giving the Model Hands

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. See the wall an LLM hits — the questions it **cannot** answer, no matter how good your prompt is
2. Write your first **tool** and describe it to the model with a **JSON schema**
3. Run **one tool call by hand**, step by step, and see exactly what the model sends back
4. Turn that single call into a **loop** — which is all an agent really is
5. Give the agent **several tools** and watch it choose between them

> **You need an OpenAI API key for this notebook.**

---

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q openai

In [ ]:
import os
import json
from datetime import datetime
from getpass import getpass

from openai import OpenAI

api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key
client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

print("Setup complete")

---

## 2. The Problem — the Model Has No Hands

An LLM is a very good text predictor. It is **not** connected to anything.

| It cannot | Because |
|---|---|
| Tell you the time | it has no clock |
| Read your files | they were never in its training data |
| Do reliable arithmetic | it predicts text, it does not calculate |
| Send an email, book a ticket, update a row | it can only produce words |

Let's watch it fail at the easiest one.

In [ ]:
# Ask the model something it genuinely cannot know
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the exact current time right now?"}]
)

print("Model says :", response.choices[0].message.content)
print("Actually   :", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Whatever it said, it either refused or guessed. It has **no way to check**.

> The model is smart, but it is sealed in a box. What it needs is **hands** - functions it can ask
> you to run on its behalf. That is what a **tool** is.

---

## 3. What Is a Tool Call?

Here is the part that surprises everyone the first time:

> **The model never runs your code.** It only ever *asks* you to run it, and waits for the answer.

Every tool call is the same five steps:

```
  1. YOU    send the question AND a list of available tools
  2. MODEL  replies "don't answer yet - please run get_current_time()"
  3. YOU    run the function in your own Python
  4. YOU    send the result back
  5. MODEL  uses that result to write the final answer
```

Steps 3 and 4 are ordinary Python. There is no magic anywhere in this list.

| Step | Who does it | What travels |
|---|---|---|
| 1 | your code | messages + tool schemas |
| 2 | the model | a **request**: tool name + arguments |
| 3 | **your code** | nothing - you just call the function |
| 4 | your code | the result, as a message |
| 5 | the model | the final answer |

That is the whole idea. The rest of this notebook is detail.

---

## 4. Your First Tool

A tool is just a normal Python function. Nothing special about it.

In [ ]:
def get_current_time() -> dict:
    """Return the current date and time."""
    now = datetime.now()
    print(f"  [tool ran] get_current_time()")
    return {"time": now.strftime("%Y-%m-%d %H:%M:%S"), "weekday": now.strftime("%A")}

In [ ]:
# Test it yourself first - always know your tool works before the model touches it
get_current_time()

---

## 5. The Schema — the Menu the Model Orders From

The function exists in Python. The model has never heard of it.

The bridge is a **JSON schema**. Think of it as a restaurant menu: it lists what is available, what
each dish is, and what you can customise.

```
{
  "name": "get_current_time",       <- must match the Python function name EXACTLY
  "description": "Get the ...",     <- WHEN should the model reach for this?
  "parameters": {                   <- what arguments does it take?
      "type": "object",
      "properties": { },
      "additionalProperties": False
  }
}
```

⚠️ **The `description` is a prompt, not documentation.** It is the only thing the model reads when
deciding whether this tool is relevant. A vague description is a tool that never gets called.

In [ ]:
get_current_time_schema = {
    "name": "get_current_time",
    "description": "Get the current date and time. Use this for anything about 'today', 'now', or dates.",
    "parameters": {
        "type": "object",
        "properties": {},
        "additionalProperties": False,
    },
}

# The API wants each tool wrapped like this
tools = [{"type": "function", "function": get_current_time_schema}]

print("Tools available to the model:", [t["function"]["name"] for t in tools])

---

## 6. One Tool Call, Step by Step

We will do the five steps **one cell at a time**, so you can see exactly what comes back.

### Step 1 and 2 — ask, and see what the model requests

In [ ]:
messages = [{"role": "user", "content": "What time is it right now?"}]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,          # <- the only new argument
)

choice = response.choices[0]

print("finish_reason :", choice.finish_reason)
print("content       :", choice.message.content)
print("tool_calls    :", choice.message.tool_calls)

Read that output carefully:

- `finish_reason` is **`"tool_calls"`**, not `"stop"`. The model is saying *"I am not done - I need
  something first."*
- `content` is **empty**. It did not answer, because it cannot yet.
- `tool_calls` holds the **request**: which function, and what arguments.

> Nothing has run. The model asked. That is all it can do.

### Step 3 — run the function yourself

In [ ]:
tool_call = choice.message.tool_calls[0]

name = tool_call.function.name
args = json.loads(tool_call.function.arguments)     # arguments arrive as a JSON *string*

print("The model asked for :", name)
print("with arguments      :", args)

result = get_current_time()                          # YOUR code runs YOUR function
print("Your function returned:", result)

### Step 4 and 5 — send the result back and get the real answer

In [ ]:
# Append what the model asked for...
messages.append(choice.message)

# ...and what your code found out. The tool_call_id must match, or the API rejects it.
messages.append({
    "role": "tool",
    "content": json.dumps(result),
    "tool_call_id": tool_call.id,
})

# Ask again - now it has what it needs
final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

print("finish_reason :", final.choices[0].finish_reason)
print("answer        :", final.choices[0].message.content)

**That took two API calls, not one.** One to request the tool, one to turn the result into a
sentence. Worth remembering: a two-tool answer costs three calls, and every call resends everything
said so far - so an agent is not cheap the way a single call is.

You have now done a complete tool call by hand. Everything from here is automation.

---

## 7. From One Call to a Loop — That Is an Agent

Doing that by hand worked for one tool call. But what if the model needs **two** tools? Or needs to
look at the first result before deciding what to ask for next?

Then you put it in a **loop** and let the model decide when it is finished.

```
        AGENT  =  LLM  +  TOOLS  +  LOOP

        LLM     (the brain)    decides what to do next
        TOOLS   (the hands)    functions it can request
        LOOP    (the process)  keeps going until the model says stop
```

`finish_reason` is the entire control flow:

| `finish_reason` | The model means | Your code does |
|---|---|---|
| `"tool_calls"` | "I need something first" | run the tools, append results, **call again** |
| `"stop"` | "I'm done - here's the answer" | return it to the user |

> There is no "agent" object anywhere below. **An agent is a `while` loop around an API call.**

In [ ]:
def handle_tool_calls(tool_calls):
    """Run every tool the model asked for and package the results as messages."""
    results = []
    for tool_call in tool_calls:                  # the model can request several at once
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        fn = globals().get(name)                  # look the function up by name
        result = fn(**args) if fn else {"error": f"Unknown tool: {name}"}

        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id,         # must match its request
        })
    return results

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant. Use the tools available to you. "
    "Never guess a date and never do arithmetic in your head."
)

MAX_ITERATIONS = 6        # a cap, not decoration - see the warning below


def run_agent(question, history=None, verbose=True):
    """The whole agent: a loop around a chat completion."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += history or []
    messages.append({"role": "user", "content": question})

    for step in range(MAX_ITERATIONS):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        choice = response.choices[0]

        if verbose:
            print(f"  [pass {step + 1}] finish_reason = {choice.finish_reason}")

        if choice.finish_reason != "tool_calls":
            return choice.message.content                 # "stop" - we are done

        messages.append(choice.message)                   # what it asked for
        messages.extend(handle_tool_calls(choice.message.tool_calls))   # what you found out

    return "I couldn't finish that within the step limit."

In [ ]:
# The same question as section 6 - now in one line
print(run_agent("What time is it right now?"))

---

## 8. A Second Tool — Watch It Choose

One tool is not interesting. The moment there are two, the model has to **decide**.

In [ ]:
def calculate(expression: str) -> dict:
    """Evaluate a Python arithmetic expression."""
    print(f"  [tool ran] calculate({expression!r})")
    # WARNING: eval() on a string the MODEL wrote is a remote-code-execution hole.
    # It is one line here so the lesson stays on the loop. In anything real, use
    # ast.literal_eval or a maths parser. Ask of every tool: what is the worst
    # call it could make?
    return {"result": eval(expression)}


tools.append({"type": "function", "function": {
    "name": "calculate",
    "description": "Evaluate an arithmetic expression such as '47853 * 1942'. Use for any calculation.",
    "parameters": {
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "A Python arithmetic expression"}},
        "required": ["expression"],
        "additionalProperties": False,
    },
}})

print("Tools available now:", [t["function"]["name"] for t in tools])

**Before you run each of the next three cells, predict how many passes it will take.**

In [ ]:
# Needs the clock
print(run_agent("What day of the week is it today?"))

In [ ]:
# Needs the calculator - try doing this one in your head
print(run_agent("What is 47853 multiplied by 1942?"))

In [ ]:
# Needs nothing - the model already knows this, so it never calls a tool
print(run_agent("What is the capital of France?"))

### The question we started with

Remember the opening question — *"I'm resigning today, what's my last working day?"* Retrieval could
never answer it, because the model has no clock and cannot do date arithmetic.

Now it has both. Watch it use **two tools in sequence**: first the clock, then the calculator — and
it works out that it needs the date *before* it can do the arithmetic.

In [ ]:
# Two tools, in order, on the question that beat us at the start
print(run_agent("My notice period is 2 months. If I resign today, what is my last working day?"))

**Nobody wrote a plan.** There is no `plan()` function anywhere in this notebook.

At every pass the model looks at what it now knows and asks "what is the next thing I need?" That
pattern has a name — **ReAct**: reason, act, observe, repeat.

And notice what we **never** wrote: an `if` statement saying *"if the user asks about time, call
`get_current_time`"*.

> The routing logic lives in English, inside the tool **descriptions**. That is why writing a good
> description is a real engineering skill and not a formality.

---

## Key Takeaways

1. **The model never runs your code.** It requests a tool; your code runs it and sends the result
   back. Five steps, and three of them are ordinary Python.

2. **An agent is a `while` loop around an API call.** LLM + tools + loop, ended by `finish_reason`.
   Every agent framework you will ever meet is this, with better error handling.

3. **The tool description is a prompt.** It is the only thing the model reads when choosing. Most
   "my agent ignores my tool" bugs are one badly written sentence.

4. **A one-tool question costs two API calls.** One to ask for the tool, one to use the result.

5. **The model decides.** With several tools available it picks the ones the question needs — and
   picks none at all when it already knows the answer.

6. **Never `while True`.** A loop with no cap around a paid API call is an invoice waiting to happen.

### Concept Map

```
  QUESTION
      |
      v
  +---------+   finish_reason == "tool_calls"   +--------------------+
  |   LLM   | --------------------------------> |  get_current_time  |
  | (brain) |                                   |  calculate         |
  |         | <-------------------------------- |  ...your tools     |
  +---------+           tool results            +--------------------+
      |
      | "stop"
      v
   ANSWER
```

### Quick Reference

| Idea | The one-liner |
|---|---|
| **Tool** | a normal Python function the model can ask you to run |
| **Schema** | name + **description** + parameters; the description does the routing |
| **Who runs it** | your code, always - the model only ever asks |
| **`tool_call_id`** | every request needs exactly one matching `role: "tool"` reply |
| **`finish_reason`** | `"tool_calls"` keep going, `"stop"` return the answer |
| **Agent** | LLM + tools + loop; the model decides when to stop |
| **`MAX_ITERATIONS`** | the difference between a bug and an invoice |

### 🏠 Homework

1. **A tool of your own.** Write a third tool that does something the model cannot do, give it a
   clear description, and ask a question that needs it.
2. **Break a description.** Change one tool's description to something vague and re-run a question
   that needs it. What happens, and what does that tell you about where the routing lives?
3. **Force a two-tool answer.** Write one question that cannot be answered without both tools, and
   count how many `[pass N]` lines print.

### 📚 Resources

- [OpenAI — function calling](https://platform.openai.com/docs/guides/function-calling)
- [Anthropic — Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)

---

**Next:** notebook 2 rebuilds this agent as a **graph** with LangGraph — where the loop becomes
something you can draw, pause and resume.